# LLM-Lab on Colab (quickstart)

A thin wrapper to run the project on a Colab GPU while **persisting artifacts to Google Drive**.

The project (code + data) runs entirely on the **local, fast** filesystem; Drive is
used only as the destination for synced artifacts. We mirror the run directory to
Drive periodically (and at the end), so the training loop never pays Drive latency.

1. Mount Drive
2. Configure + install (local project)
3. (once) Prepare data
4. Train (with a background sync to Drive)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import sys
import sysconfig
from pathlib import Path

# --- configuration ---------------------------------------------------------
# Everything runs on the LOCAL (fast) filesystem; Drive only receives synced artifacts.
PROJECT_PATH = Path("/content/LLM-Lab")                 # the repo, cloned locally
DRIVE_ROOT   = Path("/content/drive/MyDrive/LLM-Lab")   # artifact destination only
CONFIG       = "configs/experiments/scaling_model/xl3.yaml"

os.chdir(PROJECT_PATH)   # relative config paths resolve here (skip if cwd is already the project)
print("cwd:   ", Path.cwd())
print("config:", CONFIG)
print("drive: ", DRIVE_ROOT)

# --- dependencies (torch is preinstalled on Colab) -------------------------
!pip install -q datasets tokenizers tensorboard pyyaml tqdm matplotlib

# --- make `llm_lab` importable without PYTHONPATH --------------------------
# 1) this kernel:
sys.path.insert(0, str(PROJECT_PATH / "src"))
# 2) `!python ...` subprocesses (scripts/train.py) via a .pth in site-packages:
site_packages = Path(sysconfig.get_paths()["purelib"])
(site_packages / "llm_lab_src.pth").write_text(str(PROJECT_PATH / "src") + "\n")

import llm_lab
print("llm_lab:", llm_lab.__file__)
!python -c "import llm_lab; print('subprocess sees:', llm_lab.__file__)"
!python -c "import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
# Data: prepared locally (download + tokenize), persisted on Drive.
# Pull from Drive if already prepared there; otherwise prepare, then push.
import subprocess
from llm_lab.config import load_config
from llm_lab.utils.sync import sync_tree

data = load_config(CONFIG)["data"]
LOCAL_DATA = PROJECT_PATH / data["processed_dir"]
DRIVE_DATA = DRIVE_ROOT / data["processed_dir"]
LOCAL_TOK  = (PROJECT_PATH / data["tokenizer"]["output"]).parent
DRIVE_TOK  = (DRIVE_ROOT / data["tokenizer"]["output"]).parent

if (LOCAL_DATA / "train.pt").exists():
    print("data already local:", LOCAL_DATA)
elif (DRIVE_DATA / "train.pt").exists():
    print("pulling data from Drive:", sync_tree(DRIVE_DATA, LOCAL_DATA))
    sync_tree(DRIVE_TOK, LOCAL_TOK)
else:
    subprocess.run([sys.executable, "scripts/prepare_data.py", "--config", CONFIG], check=True)
    print("pushing data to Drive:", sync_tree(LOCAL_DATA, DRIVE_DATA))
    sync_tree(LOCAL_TOK, DRIVE_TOK)

In [ ]:
import threading

from llm_lab.config import load_config
from llm_lab.utils.sync import sync_tree

cfg = load_config(CONFIG)
name = f"{cfg['experiment']}/{cfg['run_name']}"
LOCAL_RUN = PROJECT_PATH / "artifacts" / "runs" / name   # fast, local
DRIVE_RUN = DRIVE_ROOT / "artifacts" / "runs" / name     # persistent, on Drive

# Resume across sessions: pull the run down from Drive if it is not local yet.
if not LOCAL_RUN.exists() and DRIVE_RUN.exists():
    print("restoring from Drive:", sync_tree(DRIVE_RUN, LOCAL_RUN))

# Background mirror to Drive every 10 minutes, so a disconnect loses little.
stop = threading.Event()

def _sync_loop():
    while not stop.wait(600):
        try:
            print("[sync]", sync_tree(LOCAL_RUN, DRIVE_RUN), flush=True)
        except Exception as exc:  # keep training even if a sync hiccups
            print("[sync] failed:", exc, flush=True)

threading.Thread(target=_sync_loop, daemon=True).start()

# Train (everything local; resume is automatic if a checkpoint exists).
!python scripts/train.py --config {CONFIG}

# Final sync.
stop.set()
print("[sync]", sync_tree(LOCAL_RUN, DRIVE_RUN))